In [1]:
# importing necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# retrieve the data
df = pd.read_csv('../../data/raw/numbers/the_numbers_worldwide_all_years.csv')
df.head(5)

,rank,title,worldwide_box_office,domestic_box_office,international_box_office,domestic_share,url,title_url
0,1,Avengers: Endgame,"$2,721,303,922","$862,173,000","$1,859,130,922",31.7%,https://www.the-numbers.com/movie/Avengers-End...,Avengers: Endgame (2019)
1,2,The Lion King,"$1,661,454,403","$543,638,043","$1,117,816,360",32.7%,https://www.the-numbers.com/movie/Lion-King-Th...,The Lion King (2019)
2,3,Frozen II,"$1,451,653,316","$477,373,578","$974,279,738",32.9%,https://www.the-numbers.com/movie/Frozen-II-(2...,Frozen II (2019)
3,4,Spider-Man: Far From Home,"$1,132,298,674","$391,362,492","$740,936,182",34.6%,https://www.the-numbers.com/movie/Spider-Man-F...,Spider-Man: Far From Home (2019)
4,5,Captain Marvel,"$1,129,576,094","$426,829,839","$702,746,255",37.8%,https://www.the-numbers.com/movie/Captain-Marv...,Captain Marvel (2019)


In [3]:
df.columns

Index(['rank', 'title', 'worldwide_box_office', 'domestic_box_office',
       'international_box_office', 'domestic_share', 'url', 'title_url'],
      dtype='str')

In [4]:
df["url"][1]

'https://www.the-numbers.com/movie/Lion-King-The-(Live-Action)-(2019)'

each url is of format: "https://www.the-numbers.com/movie/{movie-full-name-(year)}"

some urls contain the title in a false format: Odyssey the (instead of the odyssey)

that's why we changed the scraping by retrieving the h1 tag which contains the title(year)

let's treat each column separately

In [5]:
df["title"][12]
# keep in mind some titles are not complete so we need to use the url column

'The Wandering Earth (流浪地球)'

In [6]:
df.dtypes

rank                        int64
title                         str
worldwide_box_office          str
domestic_box_office           str
international_box_office      str
domestic_share                str
url                           str
title_url                     str
dtype: object

In [7]:
# let's treat the easy columns representing the budget and the gross worldwide
df["domestic_box_office"] = df["domestic_box_office"].str.replace("$", "").str.replace(",", "")
df["worldwide_box_office"] = df["worldwide_box_office"].str.replace("$", "").str.replace(",", "")
df["international_box_office"] = df["international_box_office"].str.replace("$", "").str.replace(",", "")


In [8]:
# df.head(5)
# turn those columns into floats
df["domestic_box_office"] = df["domestic_box_office"].astype(float)
df["worldwide_box_office"] = df["worldwide_box_office"].astype(float)
df["international_box_office"] = df["international_box_office"].astype(float)

In [9]:
# treating the domestic share column: removing the percentage sign and turning it into a float
df["domestic_share"] = df["domestic_share"].str.replace("%", "").astype(float)

In [10]:
df.dtypes

rank                          int64
title                           str
worldwide_box_office        float64
domestic_box_office         float64
international_box_office    float64
domestic_share              float64
url                             str
title_url                       str
dtype: object

In [11]:
df["title_url"]

0                  Avengers: Endgame (2019)
1                      The Lion King (2019)
2                          Frozen II (2019)
3          Spider-Man: Far From Home (2019)
4                     Captain Marvel (2019)
                       ...                 
795    Posledniy bogatyr. Koren’ Zla (2021)
796                   Licorice Pizza (2021)
797                The Little Things (2021)
798                       The Unholy (2021)
799                    The Last Duel (2021)
Name: title_url, Length: 800, dtype: str

In [12]:
# we will extract the year from the title_url column, and the correct title in title_url column
# the format of title_url is : title (year)
# even if all movies are released in 2026, the column year will only have one unique value, but we will keep it for the sake of consistency with other datasets

df["year"] = df["title_url"].str.extract(r'\((\d{4})\)') # regex here means: find a 4 digit number between parentheses
df["title_url"] = df["title_url"].str.extract(r'^(.*?)(?=\s*\(\d{4}\))') # regex here means: find everything before the year in parentheses


In [13]:
df["year"] = df["year"].astype(int)
df["year"].unique()


array([2019, 2020, 2022, 2023, 2024, 2025, 2026, 2021])

In [14]:
# verification 
df[["year", "title_url"]].head(10)

,year,title_url
0,2019,Avengers: Endgame
1,2019,The Lion King
2,2019,Frozen II
3,2019,Spider-Man: Far From Home
4,2019,Captain Marvel
5,2019,Joker
6,2019,Toy Story 4
7,2019,Star Wars: The Rise of Skywalker
8,2019,Aladdin
9,2019,Jumanji: The Next Level


In [15]:
# we can drop the title column since we have the title_url column which is more accurate
# we actually replace its content by title_url and drop the title_url column
df["title"] = df["title_url"]
df.drop(columns=["title_url"], inplace=True) # inplace=True without creating a new df

In [16]:
df["title"]

0                  Avengers: Endgame
1                      The Lion King
2                          Frozen II
3          Spider-Man: Far From Home
4                     Captain Marvel
                   ...              
795    Posledniy bogatyr. Koren’ Zla
796                   Licorice Pizza
797                The Little Things
798                       The Unholy
799                    The Last Duel
Name: title, Length: 800, dtype: str

In [17]:
df.columns

Index(['rank', 'title', 'worldwide_box_office', 'domestic_box_office',
       'international_box_office', 'domestic_share', 'url', 'year'],
      dtype='str')

In [18]:
# lets put the year after the title column for better viz
df = df[["rank","title", "year", "domestic_box_office", "international_box_office", "worldwide_box_office", "domestic_share", "url"]]
print(df.head(5))

   rank                      title  year  domestic_box_office  \
0     1          Avengers: Endgame  2019          862173000.0   
1     2              The Lion King  2019          543638043.0   
2     3                  Frozen II  2019          477373578.0   
3     4  Spider-Man: Far From Home  2019          391362492.0   
4     5             Captain Marvel  2019          426829839.0   

   international_box_office  worldwide_box_office  domestic_share  \
0              1.859131e+09          2.721304e+09            31.7   
1              1.117816e+09          1.661454e+09            32.7   
2              9.742797e+08          1.451653e+09            32.9   
3              7.409362e+08          1.132299e+09            34.6   
4              7.027463e+08          1.129576e+09            37.8   

                                                 url  
0  https://www.the-numbers.com/movie/Avengers-End...  
1  https://www.the-numbers.com/movie/Lion-King-Th...  
2  https://www.the-numbers.co

In [19]:
# saving the dataframe to "../../data/processed/the_numbers_worldwide_2026_cleaned.csv"
df.to_csv("../../data/processed/the_numbers_worldwide_2026_cleaned.csv", index=False)